In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Evaluation Data

Load the saved LSTM test evaluation metrics for subsequent analysis.

In [2]:
# Load LSTM test evaluation metrics
metrics_path = "../data/lstm/result/lstm_test_metrics.csv"

lstm_test_metrics = pd.read_csv(metrics_path)

# Display the evaluation metrics
display(lstm_test_metrics)

,Metric,Value
0,MAE,1.337059e+04
1,MSE,2.057336e+08
2,RMSE,1.434342e+04
3,R2,3.189571e-01


## 2. Load Gemini Insights and Retrieved News

Load the generated trading insights and retrieved news evidence for subsequent evaluation.

In [3]:
# Define file paths
insights_path = Path("../data/llm/bitcoin_llm_insights_with_urls.txt")
news_path = Path("../data/news/processed/bitcoin_news_retrieved_top10.csv")

# Load Gemini trading insights
with open(insights_path, "r", encoding="utf-8") as file:
    gemini_insights = file.read()

# Load retrieved news
retrieved_news = pd.read_csv(news_path)

# Display loading results
print("Gemini insights loaded successfully.")
print(f"Insights length: {len(gemini_insights)} characters")

print("\nRetrieved news loaded successfully.")
print(f"Number of news articles: {len(retrieved_news)}")

# Preview the data
print("\nGemini Insights Preview:")
print(gemini_insights[:1000])

print("\nRetrieved News Columns:")
print(retrieved_news.columns.tolist())

display(retrieved_news.head())

Gemini insights loaded successfully.
Insights length: 11165 characters

Retrieved news loaded successfully.
Number of news articles: 10

Gemini Insights Preview:
Bitcoin LLM Insights

1. Forecasting Result

- Forecast date: 2026-10-29
- Predicted Bitcoin closing price: 78,207.90 USD
- Note: This figure is a quantitative model prediction and does not represent an actual or guaranteed price.

2. Confidence

Evidence surrounding Bitcoin's trajectory is mixed, reflecting a cooling bullish phase and division among market analysts. On-chain metrics, such as CryptoQuant's Bull Score Index falling from 90 to 60, suggest that momentum is slowing. While some analysts view current pullbacks as a buying opportunity, others warn of potential capitulation if key technical thresholds are broken. Broader macroeconomic factors—such as inflation risks, potential central bank tightening, and upcoming corporate earnings—introduce further volatility, leaving the market's immediate direction uncertain.

3. 

,similarity_rank,similarity_score,title,source,published_at_utc,link,summary,content,retrieval_text,content_status
0,1,0.622446,"Bitcoin Slips Ahead Of Fed Week, Big Tech Earn...",Stocktwits,2026-10-26 10:21:09+00:00,https://finance.yahoo.com/markets/crypto/artic...,Analyst Michaël van de Poppe pointed to BTC’s ...,"Bitcoin slipped below $80,000 on Monday, dippi...","Title: Bitcoin Slips Ahead Of Fed Week, Big Te...",summary_and_content
1,2,0.619871,Bitcoin On Track To Post Weakest October In Si...,Stocktwits,2026-10-28 09:22:33+00:00,https://finance.yahoo.com/markets/crypto/artic...,Inflation risks could complicate the outlook f...,"Bitcoin traded around $80,800 on Wednesday, do...",Title: Bitcoin On Track To Post Weakest Octobe...,summary_and_content
2,3,0.602598,"Crypto Market Today, Oct. 28: Bitcoin and Altc...",Motley Fool,2026-10-28 21:30:01+00:00,https://finance.yahoo.com/markets/crypto/artic...,"Bitcoin extended its recovery above $81,000 as...","As of early evening on Oct. 28, Bitcoin (CRYPT...","Title: Crypto Market Today, Oct. 28: Bitcoin a...",summary_and_content
3,4,0.600868,"Bitcoin little-changed at $80.4k as oil, rate ...",Investing.com,2026-10-26 06:54:50+00:00,https://finance.yahoo.com/markets/crypto/artic...,Investing.com-- Bitcoin moved slightly lower o...,Investing.com-- Bitcoin moved slightly lower o...,Title: Bitcoin little-changed at $80.4k as oil...,summary_and_content
4,5,0.592786,Bitcoin rises slightly after Fed holds interes...,Investing.com,2026-10-28 06:49:56+00:00,https://finance.yahoo.com/markets/crypto/artic...,Investing.com -- Bitcoin was slightly higher o...,Investing.com -- Bitcoin was slightly higher o...,Title: Bitcoin rises slightly after Fed holds ...,summary_and_content


## Section 3: Explainability Evaluation

In [4]:
# Load the project API key
env_path = Path("../data/env/.env")
load_dotenv(env_path)

api_key = os.getenv("GEMINI_API_KEY")
assert api_key, f"GEMINI_API_KEY was not found. Check: {env_path.resolve()}"

# Initialise Gemini client
client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-3.5-flash-lite"

print("Gemini evaluator configured.")
print("Model:", MODEL_ID)
print("API key loaded:", bool(api_key))

Gemini evaluator configured.
Model: gemini-3.5-flash-lite
API key loaded: True


In [5]:
# Define the explainability evaluation criteria

evaluation_criteria = {
    "Relevance": {
        1: "The insights are unrelated to the LSTM forecast and Bitcoin market context.",
        2: "The insights have limited relevance to the forecast or market context.",
        3: "The insights are partially relevant but contain some unrelated information.",
        4: "The insights are relevant to the forecast and Bitcoin market context.",
        5: "The insights are highly relevant and closely connect the forecast with the market context."
    },

    "Clarity": {
        1: "The insights are very difficult to understand.",
        2: "The insights are poorly structured and difficult to follow.",
        3: "The insights are understandable but contain some unclear explanations.",
        4: "The insights are clear and logically organised.",
        5: "The insights are very clear, concise, and logically organised."
    },

    "Usefulness": {
        1: "The insights provide almost no useful information for understanding the forecast.",
        2: "The insights provide limited market context or risk information.",
        3: "The insights provide some useful context but have noticeable limitations.",
        4: "The insights provide useful market context and relevant risk information.",
        5: "The insights provide highly useful context for interpreting the forecast and understanding potential risks."
    },

    "Faithfulness": {
        1: "Major claims contradict the provided forecast or available evidence.",
        2: "Several important claims are inconsistent with the provided forecast or evidence.",
        3: "Some claims are supported, but other important claims lack sufficient evidence.",
        4: "Most important claims are consistent with the provided forecast and available evidence.",
        5: "All major verifiable claims are consistent with the provided forecast and available evidence."
    },

    "Completeness": {
        1: "The insights omit most essential information needed to interpret the forecast.",
        2: "The insights cover only a small portion of the relevant information.",
        3: "The insights cover the main points but omit some important information.",
        4: "The insights adequately cover the forecast interpretation, market context, and relevant risks.",
        5: "The insights comprehensively cover the relevant forecast interpretation, market context, and risks without requiring unavailable information."
    }
}

print("Evaluation criteria defined successfully.")
print("Dimensions:", list(evaluation_criteria.keys()))

Evaluation criteria defined successfully.
Dimensions: ['Relevance', 'Clarity', 'Usefulness', 'Faithfulness', 'Completeness']


In [6]:
# Load daily prediction records
forecast_path = Path(
    "../data/lstm/processed/bitcoin_daily_predictions.csv"
)

forecast_df = pd.read_csv(forecast_path)

# Ensure forecast dates are stored as datetime values
forecast_df["forecast_date"] = pd.to_datetime(
    forecast_df["forecast_date"],
    errors="coerce"
)

# Remove rows with invalid forecast dates
forecast_df = forecast_df.dropna(
    subset=["forecast_date"]
)

# Select the latest forecast record automatically
latest_forecast_date = forecast_df["forecast_date"].max()

forecast_row = forecast_df.loc[
    forecast_df["forecast_date"] == latest_forecast_date
].iloc[-1]

# Convert the selected record into a dictionary
forecast_evidence = {
    "forecast_date": latest_forecast_date.strftime("%Y-%m-%d"),
    "prediction_created_at": str(
        forecast_row["prediction_created_at"]
    ),
    "predicted_close": float(forecast_row["predicted_close"]),
    "actual_close": (
        None
        if pd.isna(forecast_row["actual_close"])
        else float(forecast_row["actual_close"])
    ),
    "prediction_error": (
        None
        if pd.isna(forecast_row["prediction_error"])
        else float(forecast_row["prediction_error"])
    ),
    "absolute_error": (
        None
        if pd.isna(forecast_row["absolute_error"])
        else float(forecast_row["absolute_error"])
    )
}

print("Original LSTM forecast evidence loaded successfully.")
print(forecast_evidence)

Original LSTM forecast evidence loaded successfully.
{'forecast_date': '2026-10-29', 'prediction_created_at': '2026-10-29 14:25:12', 'predicted_close': 78207.8984375, 'actual_close': None, 'prediction_error': None, 'absolute_error': None}


In [7]:
# Evaluate Gemini-generated investment insights
# Prepare independent evidence for Faithfulness evaluation

# Prepare the evaluation criteria
criteria_text = json.dumps(
    evaluation_criteria,
    indent=2,
    ensure_ascii=False
)

# Prepare the original LSTM forecast record
forecast_evidence_text = json.dumps(
    forecast_evidence,
    indent=2,
    ensure_ascii=False
)

# Prepare retrieved news evidence
news_evidence_records = []

for _, row in retrieved_news.iterrows():
    news_evidence_records.append({
        "title": str(row.get("title", "")),
        "source": str(row.get("source", "")),
        "published_at_utc": str(row.get("published_at_utc", "")),
        "link": str(row.get("link", "")),
        "summary": str(row.get("summary", "")),
        "content": str(row.get("content", ""))
    })

news_evidence_text = json.dumps(
    news_evidence_records,
    indent=2,
    ensure_ascii=False
)

# Build the evaluation prompt
evaluation_prompt = f"""
You are an evaluator assessing the explainability of cryptocurrency
investment insights generated by an LLM.

Evaluate the Bitcoin investment insights using the five dimensions
and scoring criteria provided below.

SCORING DIMENSIONS AND CRITERIA:
{criteria_text}

BITCOIN INVESTMENT INSIGHTS TO EVALUATE:
{gemini_insights}

ORIGINAL LSTM FORECAST RECORD:
{forecast_evidence_text}

RETRIEVED NEWS EVIDENCE:
{news_evidence_text}

GENERAL INSTRUCTIONS:
1. Assign an integer score from 1 to 5 for each dimension.
2. Follow the provided scoring criteria consistently.
3. Provide a concise justification for every score.
4. Evaluate the actual content of the supplied investment insights.
5. Do not assume that missing information exists.
6. Do not treat an investment recommendation as correct merely
   because it is confidently stated.
7. Do not assume that a claim is true simply because it appears
   in the investment insights.
8. Use the original forecast record and retrieved news as evidence
   when assessing Faithfulness.
9. Distinguish between information supported by the supplied evidence
   and information that cannot be verified from the available evidence.
10. Do not claim to have independently verified the original news
    publisher's website. Only the supplied news evidence is available
    for this evaluation.
11. Return valid JSON only. Do not include Markdown code fences.

FAITHFULNESS SCORING GUIDANCE:
- Score 1: Major claims contradict the original forecast or available
  evidence.
- Score 2: Several important claims are inconsistent with the
  original forecast or available evidence.
- Score 3: Some claims are supported, but other important claims lack
  sufficient evidence or cannot be verified.
- Score 4: Most important claims are consistent with the original
  forecast and available evidence, with limited uncertainty.
- Score 5: All major verifiable claims are consistent with the original
  forecast and available evidence, and no material unsupported claims
  are identified.

For Faithfulness:
- Check forecast dates and predicted prices against the original
  LSTM forecast record.
- Check factual news-related claims against the retrieved news
  evidence.
- Consider whether dates, figures, events, and source attributions
  are supported.
- Do not award a high score solely because claims sound plausible.
- If the available evidence is insufficient, reflect this limitation
  in the score and justification.
- A Faithfulness score does not by itself establish that every claim
  is factually correct.

Use exactly this JSON structure:
{{
    "Relevance": {{
        "score": 1,
        "reason": "Explanation of the score"
    }},
    "Clarity": {{
        "score": 1,
        "reason": "Explanation of the score"
    }},
    "Usefulness": {{
        "score": 1,
        "reason": "Explanation of the score"
    }},
    "Faithfulness": {{
        "score": 1,
        "reason": "Explanation of the score"
    }},
    "Completeness": {{
        "score": 1,
        "reason": "Explanation of the score"
    }}
}}
"""

# Request evaluation from Gemini
response = client.models.generate_content(
    model=MODEL_ID,
    contents=evaluation_prompt,
    config={
        "response_mime_type": "application/json",
        "temperature": 0
    }
)

# Parse the JSON response
evaluation_result = json.loads(response.text)

print("Explainability evaluation completed.")
print(json.dumps(evaluation_result, indent=2, ensure_ascii=False))

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"


Explainability evaluation completed.
{
  "Relevance": {
    "score": 5,
    "reason": "The insights are highly relevant, explicitly incorporating the LSTM forecast date and predicted closing price, and connecting them comprehensively with the current Bitcoin market context, macroeconomic headlines, and on-chain metrics."
  },
  "Clarity": {
    "score": 5,
    "reason": "The insights are exceptionally clear, concise, and logically organized into distinct sections including forecasting results, confidence, key market news, and potential risks, making them very easy to read and follow."
  },
  "Usefulness": {
    "score": 5,
    "reason": "The insights provide highly useful context for interpreting the LSTM forecast by detailing relevant market news, institutional behavior, macroeconomic risks, and technical support levels, offering actionable information for risk management."
  },
  "Faithfulness": {
    "score": 5,
    "reason": "All major verifiable claims, including the forecast date

In [8]:
# Save explainability evaluation results
# Convert evaluation results into a DataFrame
evaluation_df = pd.DataFrame([
    {
        "Dimension": dimension,
        "Score": details["score"],
        "Reason": details["reason"]
    }
    for dimension, details in evaluation_result.items()
])

# Calculate the average score
average_score = evaluation_df["Score"].mean()

# Add the average score as a summary row
summary_row = pd.DataFrame([{
    "Dimension": "Average",
    "Score": average_score,
    "Reason": "Average score across the five explainability dimensions."
}])

evaluation_df = pd.concat(
    [evaluation_df, summary_row],
    ignore_index=True
)

# Save the results
results_dir = Path("../data/evaluation")
results_dir.mkdir(parents=True, exist_ok=True)

evaluation_path = results_dir / "explainability_evaluation.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print("Explainability evaluation saved successfully.")
print(f"File: {evaluation_path}")
display(evaluation_df)

Explainability evaluation saved successfully.
File: ..\data\evaluation\explainability_evaluation.csv


,Dimension,Score,Reason
0,Relevance,5.0,"The insights are highly relevant, explicitly i..."
1,Clarity,5.0,"The insights are exceptionally clear, concise,..."
2,Usefulness,5.0,The insights provide highly useful context for...
3,Faithfulness,5.0,"All major verifiable claims, including the for..."
4,Completeness,5.0,The insights comprehensively cover the forecas...
5,Average,5.0,Average score across the five explainability d...


## Section 4: Baseline Comparison

### 4.1 Comparison Framework

The LSTM-only baseline provides numerical Bitcoin price forecasts.
The hybrid framework uses the same LSTM forecast and supplements it
with retrieved financial news and LLM-generated investment insights.

The comparison focuses on the additional interpretative information
provided by the hybrid framework, rather than forecasting accuracy,
because both approaches use the same underlying LSTM prediction.

In [9]:
# Compare the LSTM-only baseline with the hybrid framework

baseline_comparison = pd.DataFrame([
    {
        "Evaluation Aspect": "Price Forecasting",
        "LSTM-only Baseline": "Provides numerical Bitcoin price forecasts.",
        "Hybrid Framework": "Uses the same LSTM forecast.",
        "Comparison": "Both approaches use the same underlying forecast."
    },
    {
        "Evaluation Aspect": "Financial News Retrieval",
        "LSTM-only Baseline": "Does not retrieve financial news.",
        "Hybrid Framework": "Retrieves relevant Bitcoin market news.",
        "Comparison": "The hybrid framework adds financial news context."
    },
    {
        "Evaluation Aspect": "Market Context",
        "LSTM-only Baseline": "Does not provide LLM-generated market interpretation.",
        "Hybrid Framework": "Uses retrieved news to provide market context.",
        "Comparison": "The hybrid framework adds contextual interpretation."
    },
    {
        "Evaluation Aspect": "Potential Risk Analysis",
        "LSTM-only Baseline": "Does not generate a separate LLM-based risk analysis.",
        "Hybrid Framework": "Generates potential risk information through the LLM.",
        "Comparison": "The hybrid framework adds an explicit risk interpretation."
    },
    {
        "Evaluation Aspect": "Investment Insights",
        "LSTM-only Baseline": "Provides the numerical forecast without LLM-generated insights.",
        "Hybrid Framework": "Combines the forecast, retrieved news, and LLM-generated insights.",
        "Comparison": "The hybrid framework provides additional interpretative information."
    }
])

# Display the comparison
display(baseline_comparison)

# Save the comparison results
results_dir = Path("../data/evaluation")
results_dir.mkdir(parents=True, exist_ok=True)

comparison_path = results_dir / "baseline_comparison.csv"
baseline_comparison.to_csv(comparison_path, index=False)

print("Baseline comparison saved successfully.")
print(f"File: {comparison_path}")

,Evaluation Aspect,LSTM-only Baseline,Hybrid Framework,Comparison
0,Price Forecasting,Provides numerical Bitcoin price forecasts.,Uses the same LSTM forecast.,Both approaches use the same underlying forecast.
1,Financial News Retrieval,Does not retrieve financial news.,Retrieves relevant Bitcoin market news.,The hybrid framework adds financial news context.
2,Market Context,Does not provide LLM-generated market interpre...,Uses retrieved news to provide market context.,The hybrid framework adds contextual interpret...
3,Potential Risk Analysis,Does not generate a separate LLM-based risk an...,Generates potential risk information through t...,The hybrid framework adds an explicit risk int...
4,Investment Insights,Provides the numerical forecast without LLM-ge...,"Combines the forecast, retrieved news, and LLM...",The hybrid framework provides additional inter...


Baseline comparison saved successfully.
File: ..\data\evaluation\baseline_comparison.csv


## Section 5: Evaluation Summary

In [10]:
# Summarise the evaluation results

# Calculate the average explainability score
explainability_average = evaluation_df.loc[
    evaluation_df["Dimension"] != "Average",
    "Score"
].mean()

# Count the baseline comparison aspects
comparison_aspects = len(baseline_comparison)

# Prepare the evaluation summary
evaluation_summary = pd.DataFrame([
    {
        "Evaluation Component": "LSTM Forecasting Performance",
        "Result": "Evaluated using MAE, MSE, RMSE, and R2.",
        "Status": "Completed"
    },
    {
        "Evaluation Component": "Explainability Evaluation",
        "Result": (
            f"Single-case evaluation: "
            f"{explainability_average:.2f}/5.00 "
            "across five dimensions."
        ),
        "Status": "Completed"
    },
    {
        "Evaluation Component": "Baseline Comparison",
        "Result": (
            f"Compared LSTM-only and hybrid frameworks "
            f"across {comparison_aspects} aspects."
        ),
        "Status": "Completed"
    }
])

# Display the evaluation summary
display(evaluation_summary)

# Save the summary
summary_path = results_dir / "evaluation_summary.csv"

evaluation_summary.to_csv(
    summary_path,
    index=False
)

print("Evaluation summary saved successfully.")
print(f"File: {summary_path}")

,Evaluation Component,Result,Status
0,LSTM Forecasting Performance,"Evaluated using MAE, MSE, RMSE, and R2.",Completed
1,Explainability Evaluation,Single-case evaluation: 5.00/5.00 across five ...,Completed
2,Baseline Comparison,Compared LSTM-only and hybrid frameworks acros...,Completed


Evaluation summary saved successfully.
File: ..\data\evaluation\evaluation_summary.csv


## Section 6: Hallucination Evaluation

In [11]:
# Load the insights, retrieved news, and original LSTM predictions

# Reuse gemini_insights loaded in Section 2

# Load retrieved news
news_path = Path("../data/news/processed/bitcoin_news_retrieved_top10.csv")
retrieved_news = pd.read_csv(news_path)

# Load original daily LSTM prediction records
predictions_path = Path(
    "../data/lstm/processed/bitcoin_daily_predictions.csv"
)
daily_predictions = pd.read_csv(predictions_path)

# Extract the forecast date stated in the Gemini insights
date_match = re.search(
    r"Forecast date:\s*(\d{4}-\d{2}-\d{2})",
    gemini_insights
)

if not date_match:
    raise ValueError(
        "Forecast date was not found in the Gemini insights."
    )

forecast_date = date_match.group(1)

# Find the corresponding original LSTM prediction
matching_prediction = daily_predictions.loc[
    daily_predictions["forecast_date"].astype(str) == forecast_date
]

if matching_prediction.empty:
    raise ValueError(
        f"No LSTM prediction record found for {forecast_date}."
    )

# There should be one prediction record for the selected forecast date
if len(matching_prediction) > 1:
    raise ValueError(
        f"Multiple prediction records found for {forecast_date}. "
        "Please check the prediction CSV."
    )

original_prediction = matching_prediction.iloc[0]

# Prepare the original forecast evidence
forecast_evidence = {
    "forecast_date": forecast_date,
    "predicted_close": float(original_prediction["predicted_close"])
}

# Display loading results
print("Gemini insights loaded:", len(gemini_insights), "characters")
print("Retrieved news articles:", len(retrieved_news))
print("Forecast date:", forecast_date)
print("Original LSTM forecast:", forecast_evidence)

print("\nRetrieved news columns:")
print(retrieved_news.columns.tolist())

print("\nRetrieved news preview:")
display(
    retrieved_news[
        ["title", "source", "published_at_utc", "link"]
    ].head(10)
)

Gemini insights loaded: 11165 characters
Retrieved news articles: 10
Forecast date: 2026-10-29
Original LSTM forecast: {'forecast_date': '2026-10-29', 'predicted_close': 78207.8984375}

Retrieved news columns:
['similarity_rank', 'similarity_score', 'title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'retrieval_text', 'content_status']

Retrieved news preview:


,title,source,published_at_utc,link
0,"Bitcoin Slips Ahead Of Fed Week, Big Tech Earn...",Stocktwits,2026-10-26 10:21:09+00:00,https://finance.yahoo.com/markets/crypto/artic...
1,Bitcoin On Track To Post Weakest October In Si...,Stocktwits,2026-10-28 09:22:33+00:00,https://finance.yahoo.com/markets/crypto/artic...
2,"Crypto Market Today, Oct. 28: Bitcoin and Altc...",Motley Fool,2026-10-28 21:30:01+00:00,https://finance.yahoo.com/markets/crypto/artic...
3,"Bitcoin little-changed at $80.4k as oil, rate ...",Investing.com,2026-10-26 06:54:50+00:00,https://finance.yahoo.com/markets/crypto/artic...
4,Bitcoin rises slightly after Fed holds interes...,Investing.com,2026-10-28 06:49:56+00:00,https://finance.yahoo.com/markets/crypto/artic...
5,"Crypto Market Today, Oct. 26: Bitcoin Slips Be...",Motley Fool,2026-10-26 21:25:02+00:00,https://finance.yahoo.com/markets/crypto/artic...
6,"Is Bitcoin a Good Buy at $80,000 Right Now?",24/7 Wall St.,2026-10-26 14:56:20+00:00,https://finance.yahoo.com/markets/crypto/artic...
7,"Bitcoin up as risk sentiment, oil slip offset ...",Investing.com,2026-10-27 06:56:49+00:00,https://finance.yahoo.com/markets/crypto/artic...
8,"Bitcoin faces further downside as oil surge, A...",Investing.com,2026-10-27 13:53:17+00:00,https://finance.yahoo.com/markets/crypto/artic...
9,"$8 Trillion in Big Tech Earnings, Fed Decision...",CCN,2026-10-26 15:32:18+00:00,https://finance.yahoo.com/markets/crypto/artic...


In [12]:
# Hallucination Evaluation
# Evaluate claims against the original LSTM prediction and retrieved news.

# Prepare retrieved news evidence
news_evidence_records = []

for _, row in retrieved_news.iterrows():
    news_evidence_records.append({
        "title": str(row.get("title", "")),
        "source": str(row.get("source", "")),
        "published_at_utc": str(row.get("published_at_utc", "")),
        "link": str(row.get("link", "")),
        "summary": str(row.get("summary", "")),
        "content": str(row.get("content", ""))
    })

news_evidence_text = json.dumps(
    news_evidence_records,
    ensure_ascii=False,
    indent=2
)

# Prepare original LSTM forecast evidence
forecast_evidence_text = json.dumps(
    forecast_evidence,
    ensure_ascii=False,
    indent=2
)

# Build the evaluation prompt
hallucination_prompt = f"""
You are evaluating factual claims in Bitcoin investment insights.

Your task is to identify factual claims and assess them strictly
against the supplied original forecast record and retrieved news evidence.

GEMINI INVESTMENT INSIGHTS:
{gemini_insights}

ORIGINAL LSTM FORECAST RECORD:
{forecast_evidence_text}

RETRIEVED NEWS EVIDENCE:
{news_evidence_text}

EVALUATION RULES:

1. Extract meaningful, specific, verifiable factual claims from the
   investment insights. Avoid duplicate claims.

2. Evaluate forecast-related claims only against the ORIGINAL LSTM
   FORECAST RECORD.

3. Evaluate news-related claims against the RETRIEVED NEWS EVIDENCE.
   The investment insights themselves must never be treated as evidence.

4. For every news-related claim, identify the specific article and
   quote the relevant supporting text from its supplied title, summary,
   or content. The quotation must match the supplied evidence.

5. For every forecast-related claim, identify the relevant forecast
   field and its recorded value.

6. Do not invent quotations, sources, article details, dates, or
   numerical values.

7. Use exactly one of these categories:
   - Supported: the supplied independent evidence directly supports
     all important factual components of the claim.
   - Contradicted: the supplied independent evidence directly conflicts
     with the claim.
   - Unverifiable: the supplied evidence is insufficient to establish
     whether the claim is true.

8. If a claim contains multiple factual components, do not mark the
   entire claim Supported unless all important components are supported.

9. The absence of evidence does not establish that a claim is false.
   Use Unverifiable when evidence is insufficient.

10. Do not claim that an article or external fact has been independently
    verified beyond the supplied evidence. You are checking consistency
    with the provided records, not independently verifying the original
    publisher's website.

11. Extract claims only when meaningful factual claims are present.
    Do not force a fixed number of claims.

12. The source field must identify the relevant forecast record or
    retrieved news article. For news, include the article title or
    another identifier available in the supplied evidence.

Return valid JSON only, with this structure:

{{
  "claims": [
    {{
      "claim": "The factual claim being evaluated",
      "category": "Supported, Contradicted, or Unverifiable",
      "evidence": "The exact relevant forecast field or quotation from the supplied news evidence",
      "source": "The original forecast record or specific retrieved news article",
      "reason": "A concise explanation of why the evidence supports, contradicts, or cannot establish the claim"
    }}
  ]
}}
"""

# Call the configured Gemini model
hallucination_response = client.models.generate_content(
    model=MODEL_ID,
    contents=hallucination_prompt
)

# Parse the JSON response
response_text = hallucination_response.text.strip()

# Remove Markdown code fences if the model includes them
if response_text.startswith("```"):
    response_text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        response_text,
        flags=re.IGNORECASE
    )

hallucination_result = json.loads(response_text)

# Convert claims into a DataFrame
hallucination_df = pd.DataFrame(
    hallucination_result.get("claims", [])
)

required_columns = [
    "claim",
    "category",
    "evidence",
    "source",
    "reason"
]

for column in required_columns:
    if column not in hallucination_df.columns:
        hallucination_df[column] = ""

hallucination_df = hallucination_df[required_columns]

# Display results
print("Hallucination evaluation completed.")
print("Claims identified:", len(hallucination_df))

display(hallucination_df)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"


Hallucination evaluation completed.
Claims identified: 5


,claim,category,evidence,source,reason
0,The predicted Bitcoin closing price for Octobe...,Supported,"""predicted_close"": 78207.8984375",ORIGINAL LSTM FORECAST RECORD,The original forecast record explicitly specif...
1,"Bitcoin slipped below $80,000 ahead of the Fed...",Supported,Bitcoin (BTC) is bracing for its weakest Octob...,Stocktwits (Article 2),NaN
2,Bitcoin was on track for its weakest October i...,Supported,Bitcoin (BTC) is bracing for its weakest Octob...,Stocktwits (Article 2),NaN
3,The Federal Reserve held benchmark interest ra...,Supported,The Federal Open Market Committee (FOMC) maint...,Investing.com (Article 5),NaN
4,CryptoQuant's Bull Score Index fell from 90 to...,Supported,"CryptoQuant's Bull Score Index, which aggregat...",Stocktwits (Article 2),NaN


In [13]:
# Convert hallucination evaluation results into a DataFrame

hallucination_df = pd.DataFrame(
    hallucination_result["claims"]
)

# Display all identified claims and their evidence
display(hallucination_df)

# Summarise the classification counts
hallucination_summary = (
    hallucination_df["category"]
    .value_counts()
    .reindex(
        ["Supported", "Contradicted", "Unverifiable"],
        fill_value=0
    )
    .rename_axis("Category")
    .reset_index(name="Count")
)

print("Hallucination evaluation summary:")
display(hallucination_summary)

,claim,category,evidence,source,reason
0,The predicted Bitcoin closing price for Octobe...,Supported,"""predicted_close"": 78207.8984375",ORIGINAL LSTM FORECAST RECORD,The original forecast record explicitly specif...
1,"Bitcoin slipped below $80,000 ahead of the Fed...",Supported,Bitcoin (BTC) is bracing for its weakest Octob...,Stocktwits (Article 2),NaN
2,Bitcoin was on track for its weakest October i...,Supported,Bitcoin (BTC) is bracing for its weakest Octob...,Stocktwits (Article 2),NaN
3,The Federal Reserve held benchmark interest ra...,Supported,The Federal Open Market Committee (FOMC) maint...,Investing.com (Article 5),NaN
4,CryptoQuant's Bull Score Index fell from 90 to...,Supported,"CryptoQuant's Bull Score Index, which aggregat...",Stocktwits (Article 2),NaN


Hallucination evaluation summary:


,Category,Count
0,Supported,5
1,Contradicted,0
2,Unverifiable,0


In [14]:
# Independently check claims against the original LSTM forecast
# or the retrieved news dataset.

def normalise_text(text):
    """Normalise text for evidence matching."""
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)

    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s]", " ", text)

    # Normalise whitespace
    return " ".join(text.split())


# Prepare original retrieved news records
original_news_records = []

for _, row in retrieved_news.iterrows():
    original_news_records.append({
        "title": str(row.get("title", "")),
        "source": str(row.get("source", "")),
        "published_at_utc": str(row.get("published_at_utc", "")),
        "link": str(row.get("link", "")),
        "summary": str(row.get("summary", "")),
        "content": str(row.get("content", ""))
    })


# Prepare the original LSTM forecast record
recorded_date = pd.to_datetime(
    forecast_evidence["forecast_date"]
).date()

recorded_price = float(
    forecast_evidence["predicted_close"]
)


# Check each evaluated claim
evidence_check_results = []

for _, claim_row in hallucination_df.iterrows():

    claim = str(claim_row.get("claim", ""))
    category = str(claim_row.get("category", ""))
    source = str(claim_row.get("source", ""))
    evidence = str(claim_row.get("evidence", ""))

    source_lower = source.lower()
    claim_lower = claim.lower()

    matched_articles = []
    matched_evidence_type = "Not found"

    # Default values
    evidence_status = "Not Verified"
    check_note = ""

    # 1. Check forecast-related claims
    # Forecast claims must be checked against the original LSTM record
    if (
        "forecast" in source_lower
        or "lstm" in source_lower
    ):
        recorded_date = pd.to_datetime(
            forecast_evidence["forecast_date"]
        )

        recorded_price = float(
            forecast_evidence["predicted_close"]
        )

        # Extract and normalise the date in the claim
        forecast_claim = normalise_text(claim)

        date_formats = [
            recorded_date.strftime("%Y-%m-%d"),
            recorded_date.strftime("%B %d, %Y"),
            recorded_date.strftime("%B %d %Y"),
            recorded_date.strftime("%b %d, %Y"),
            recorded_date.strftime("%b %d %Y")
        ]

        date_matches = any(
            normalise_text(date_format) in forecast_claim
            for date_format in date_formats
        )

        # Extract the price from the claim and compare numerically
        price_pattern = (
            r"(?<!\d)(?:\d{1,3}(?:,\d{3})+|\d+)"
            r"(?:\.\d+)?"
        )

        price_matches = False

        for value in re.findall(price_pattern, claim):
            try:
                claim_price = float(value.replace(",", ""))

                if abs(claim_price - recorded_price) < 0.01:
                    price_matches = True
                    break

            except ValueError:
                continue

        if date_matches and price_matches:
            evidence_status = "Verified"
            matched_evidence_type = (
                "Original LSTM forecast record"
            )
            check_note = (
                "The forecast date and predicted price match "
                "the original LSTM forecast record."
            )

        else:
            evidence_status = "Not Verified"
            matched_evidence_type = "Not found"

            missing_items = []

            if not date_matches:
                missing_items.append("forecast date")

            if not price_matches:
                missing_items.append("predicted price")

            check_note = (
                "Could not verify the following against the "
                "original LSTM forecast record: "
                + ", ".join(missing_items)
                + ". Manual review may be required."
            )
            
    # 2. Check news-related claims
    else:

        evidence_normalised = normalise_text(evidence)

        # Only use meaningful evidence text
        evidence_words = {
            word for word in evidence_normalised.split()
            if len(word) >= 4
        }

        # Avoid treating very short evidence as a reliable match
        if len(evidence_words) < 3:
            check_note = (
                "The supplied evidence is too short for reliable "
                "text matching. Manual review is required."
            )

        else:

            for article in original_news_records:

                searchable_text = normalise_text(
                    " ".join([
                        article["title"],
                        article["summary"],
                        article["content"]
                    ])
                )

                article_words = set(searchable_text.split())

                # Check for a complete evidence-text match
                exact_match = (
                    bool(evidence_normalised)
                    and evidence_normalised in searchable_text
                )

                # Calculate word overlap to identify candidate articles
                overlap_ratio = (
                    len(evidence_words & article_words)
                    / len(evidence_words)
                )

                if exact_match or overlap_ratio >= 0.70:

                    matched_articles.append({
                        "title": article["title"],
                        "source": article["source"],
                        "link": article["link"],
                        "match_type": (
                            "Exact text match"
                            if exact_match
                            else "Partial keyword match"
                        )
                    })

            if matched_articles:

                matched_evidence_type = "; ".join(
                    sorted({
                        article["match_type"]
                        for article in matched_articles
                    })
                )

                if matched_evidence_type == "Exact text match":
                    evidence_status = "Evidence Located"

                    check_note = (
                        "Matching evidence text was located in "
                        "the retrieved news dataset. The full claim "
                        "still requires contextual verification."
                    )

                else:
                    evidence_status = "Partially Matched"

                    check_note = (
                        "Potentially relevant articles were located "
                        "using keyword overlap. This does not establish "
                        "that the complete claim is supported. "
                        "Manual review is required."
                    )

            else:
                evidence_status = "Not Verified"

                check_note = (
                    "No sufficiently close evidence-text match was "
                    "located in the retrieved news dataset. This does "
                    "not prove the claim is false."
                )

    # Store the original classification and independent check separately
    evidence_check_results.append({
        "claim": claim,
        "original_category": category,
        "evidence_check_status": evidence_status,
        "matched_evidence_type": matched_evidence_type,
        "matched_articles": "; ".join(
            article["title"]
            for article in matched_articles
        ),
        "check_note": check_note
    })


# Create the evidence-checking table
evidence_check_df = pd.DataFrame(evidence_check_results)

print("Independent evidence check completed.")
print("Claims checked:", len(evidence_check_df))

display(evidence_check_df)

Independent evidence check completed.
Claims checked: 5


,claim,original_category,evidence_check_status,matched_evidence_type,matched_articles,check_note
0,The predicted Bitcoin closing price for Octobe...,Supported,Verified,Original LSTM forecast record,,The forecast date and predicted price match th...
1,"Bitcoin slipped below $80,000 ahead of the Fed...",Supported,Partially Matched,Exact text match; Partial keyword match,Bitcoin On Track To Post Weakest October In Si...,Potentially relevant articles were located usi...
2,Bitcoin was on track for its weakest October i...,Supported,Partially Matched,Exact text match; Partial keyword match,Bitcoin On Track To Post Weakest October In Si...,Potentially relevant articles were located usi...
3,The Federal Reserve held benchmark interest ra...,Supported,Partially Matched,Exact text match; Partial keyword match,Bitcoin rises slightly after Fed holds interes...,Potentially relevant articles were located usi...
4,CryptoQuant's Bull Score Index fell from 90 to...,Supported,Evidence Located,Exact text match,Bitcoin On Track To Post Weakest October In Si...,Matching evidence text was located in the retr...


In [15]:
# Save evaluation results
results_dir = Path("../data/evaluation")
results_dir.mkdir(parents=True, exist_ok=True)

# 1. Save individual hallucination claims
claims_path = (
    results_dir / "hallucination_evaluation_claims.csv"
)

hallucination_df.to_csv(
    claims_path,
    index=False
)

# 2. Save hallucination classification summary
summary_path = (
    results_dir / "hallucination_evaluation_summary.csv"
)

hallucination_summary.to_csv(
    summary_path,
    index=False
)

# 3. Save independent evidence-checking results
evidence_check_path = (
    results_dir / "independent_evidence_check.csv"
)

evidence_check_df.to_csv(
    evidence_check_path,
    index=False
)

# 4. Save explainability evaluation results
explainability_df = pd.DataFrame([
    {
        "Dimension": dimension,
        "Score": details["score"],
        "Reason": details["reason"]
    }
    for dimension, details in evaluation_result.items()
])

explainability_path = (
    results_dir / "explainability_evaluation.csv"
)

explainability_df.to_csv(
    explainability_path,
    index=False
)

# Display save status
print("All evaluation results saved successfully.")
print(f"Hallucination claims: {claims_path}")
print(f"Hallucination summary: {summary_path}")
print(f"Independent evidence check: {evidence_check_path}")
print(f"Explainability evaluation: {explainability_path}")

All evaluation results saved successfully.
Hallucination claims: ..\data\evaluation\hallucination_evaluation_claims.csv
Hallucination summary: ..\data\evaluation\hallucination_evaluation_summary.csv
Independent evidence check: ..\data\evaluation\independent_evidence_check.csv
Explainability evaluation: ..\data\evaluation\explainability_evaluation.csv
